# PDF Retrieval

At this point we have manually screened the abstracts and titles of the papers according to our inclusion/exclusion criteria. The file `screening_consolidated.xlsx` contains the results of that screening. 
Now we retrieve the PDFs of the papers that passed screening whenever possible (e.g. open access, or through the university credentials). 
This retrieval is done partially through Zotero, which can fetch PDFs from various sources. 
In some cases, when Zotero cannot find a PDF, we will need to manually retrieve it.

The work is split deliberately: **Zotero does the fetching, this notebook does the
bookkeeping.** 

| # | Who/Where | Step |
|---|---|---|
| 1 | this notebook | Emit the paste-ready identifier list and the manual queue |
| 2 | in Zotero | New collection → **Add Item by Identifier**, paste the list |
| 3 | in Zotero (with unis VPN) | Select all → **Find Available PDFs** |
| 4 | this notebook | Read the collection back, report what is still missing |
| 5 | human | Look for remaining papers manually |
| 6 | this notebook | Re-run step 4 and 5 until it converges (all papers retrieved) |

Steps 4–6 are a loop. The reconciliation is **read-only and idempotent**, it never writes
to Zotero, so run it as often as you like, including after swapping an arXiv preprint for
the published version.

Input: `../data/processed/screening/screening_consolidated.xlsx` from
[04_screening_consolidation.ipynb](04_screening_consolidation.ipynb).

Outputs, all in `../data/processed/retrieval/`:

- `zotero_identifiers.txt` — the paste list for step 2
- `retrieval_tracking.xlsx` — one row per included paper; the PRISMA *reports sought for
  retrieval / reports not retrieved* record
- `manual_worklist.html` — clickable checklist of the gaps, for step 5

The logic lives in [`src/zotero_retrieval.py`](../src/zotero_retrieval.py) rather than in
this notebook, because step 4 needs to run repeatedly and from the command line.

## Setup

The module resolves its paths from the project root, so the notebook changes directory
once. Re-running the cell is harmless.

In [44]:
import importlib
import src.zotero_retrieval
importlib.reload(src.zotero_retrieval)

<module 'src.zotero_retrieval' from '/home/m-chaves/slr_climate_nlp/src/zotero_retrieval.py'>

In [45]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.zotero_retrieval import (
    IDENTIFIERS_PATH,
    TRACKING_PATH,
    WORKLIST_PATH,
    build_identifier_plan,
    load_included,
    reconcile,
    summarise,
    write_identifier_file,
)

pd.set_option("display.max_colwidth", 70)
print(f"working directory: {Path.cwd()}")

working directory: /home/m-chaves/slr_climate_nlp


## Step 1 : get list of paper IDs to paste into Zotero

One identifier per paper, never two, or Zotero creates duplicate items for papers we hold
several ids for. The precedence is **DOI → arXiv ID**, for two reasons:

- Zotero's *Add Item by Identifier* resolves DOI, ISBN, PMID, arXiv ID, and ADS Bibcode.
  **It does not accept ACL Anthology IDs**, so those can never go in the paste list. Four
  of our five ACL papers carry a proper `10.18653/…` DOI and resolve that way regardless.
- Preferring the DOI pulls the *published* version rather than the preprint, which is what
  we want in the final bibliography.

Papers with neither a DOI nor an arXiv ID fall to the manual queue.

In [25]:
included = load_included()
plan = build_identifier_plan(included)

assert plan["paper_id"].is_unique, "duplicate paper_id"

print(f"Included papers: {len(plan)}\n")
print("Route into Zotero:")
print(plan["route"].value_counts().to_string())
print("\nIdentifier used:")
print(plan["identifier_type"].value_counts(dropna=False).to_string())

Included papers: 81

Route into Zotero:
route
identifier    77
manual         4

Identifier used:
identifier_type
doi         75
NaN          4
arxiv_id     2


In [26]:
n_written = write_identifier_file(plan)
print(f"Wrote {n_written} identifiers -> {IDENTIFIERS_PATH}\n")

manual = plan[plan["route"] == "manual"]
print(f"{len(manual)} papers have no identifier Zotero can resolve; "
      f"save these with the browser connector instead:\n")
for _, row in manual.iterrows():
    print(f"  {row['title'][:78]}")
    print(f"      {row['manual_url']}\n")

Wrote 77 identifiers -> data/processed/retrieval/zotero_identifiers.txt

4 papers have no identifier Zotero can resolve; save these with the browser connector instead:

  WorthIt: Check-worthiness Estimation of Italian Social Media Posts
      https://www.scopus.com/record/display.uri?eid=2-s2.0-105034257445&origin=resultslist

  TeamX at CheckThat! 2023: Multilingual and Multimodal Approach for Check-Worth
      https://www.scopus.com/record/display.uri?eid=2-s2.0-85175632280&origin=resultslist

  A Few Hypocrites: Few-Shot Learning and Subtype Definitions for Detecting Hypo
      https://aclanthology.org/2024.cpss-1.4/

  Exploring the Network and Topic Stability in Climate Change Deniers' Disinform
      https://www.scopus.com/record/display.uri?eid=2-s2.0-85199757520&origin=resultslist



We check the articles that only have an arXiv ID becuase they could be not peer-reviewed. We found that they are all peer-reviewed. 

In [21]:
plan[plan["identifier_type"] == "arxiv_id"]

,paper_id,title,year,route,identifier_type,identifier,doi,arxiv_id,acl_id,scopus_eid,wos_id,manual_url,scholar_url,research_area
7,7534a487c67d3408,A Multi-task Model for Sentiment Aided Stance Detection of Climate...,2022,identifier,arxiv_id,arXiv:2211.03533,NaN,2211.03533,NaN,NaN,NaN,NaN,https://scholar.google.com/scholar?q=%22A%20Multi-task%20Model%20f...,Polarization
8,ae87b61b3642db2c,Enhancing LLMs for Governance with Human Oversight: Evaluating and...,2025,identifier,arxiv_id,arXiv:2501.13802,NaN,2501.13802,NaN,NaN,NaN,NaN,https://scholar.google.com/scholar?q=%22Enhancing%20LLMs%20for%20G...,Misinformation


## Steps 2 and 3 : retrieve papers and pdfs in Zotero

**Populate the Zotero collection.** 
In the desired collection (in our case it is called "SLR_included") > File > Add by Identifier, paste the contents of `zotero_identifiers.txt`.
The box accepts one identifier per line and queues them all. Expect a few failures; anything that does not resolve will show up in step 4's gap list and can be handled manually.

**Fetch the open-access PDFs.** 
Connect to the university VPN first, so Zotero can reach subscription content as well. 
Select every item in the collection > right-click > Find Available PDFs. 

## Step 4 : check what is still missing

Reading the library needs a **read-only** Zotero API key, created at
[zotero.org/settings/keys](https://www.zotero.org/settings/keys). 
Read-only is enough: this code never writes to Zotero. 
Add both values to the existing `.env`. 

```
ZOTERO_API_KEY=...
ZOTERO_LIBRARY_ID=...
```

`ZOTERO_LIBRARY_ID` is the numeric *userID* shown on that same settings page, not the username.

Items are matched back to `paper_id` by normalized DOI, then arXiv ID, then fuzzy title the last one covers the papers Zotero titled differently from Scopus or WoS.

In [22]:
COLLECTION = "SLR_included"   # must match the collection name in Zotero exactly

tracking = reconcile(COLLECTION)
print(summarise(tracking))
print(f"\nTracking sheet : {TRACKING_PATH}")
print(f"Worklist       : {WORKLIST_PATH}")

Included papers               : 82
  present in Zotero           :  82 (100%)
  with a PDF attached         :  62 (76%)
  PDF looks like a preprint   :   2
  missing from Zotero         :   0
  in Zotero but no PDF        :  20

Tracking sheet : data/processed/retrieval/retrieval_tracking.xlsx
Worklist       : data/processed/retrieval/manual_worklist.html


### What is still missing

`manual_worklist.html` holds the same list with clickable publisher, arXiv, landing-page,
and Google Scholar links. Open it next to Zotero while on the VPN and work top to bottom.

In [23]:
gaps = tracking[~tracking["has_pdf"]]
print(f"{len(gaps)} of {len(tracking)} papers still have no PDF\n")

for reason, group in gaps.groupby(gaps["in_zotero"].map({True: "in Zotero, no PDF",
                                                         False: "not in Zotero at all"})):
    print(f"{reason} ({len(group)}):")
    for _, row in group.iterrows():
        print(f"  {row['year']}  {row['title'][:72]}")
    print()

20 of 82 papers still have no PDF

in Zotero, no PDF (20):
  2020  Opinion-based homogeneity on youtube: Combining sentiment and social net
  2023  Social Noise on Social Media and Users Perception of Global Warming
  2024  United in criticism: The discursive politics and coalitions of Australia
  2025  Social media discourse as a window into energy transition: Analyzing pub
  2025  Good and Bad Science? Spanish-Speaking YouTube Users’ Comments on Scienc
  2025  The impact of presentation modalities on perceived accuracy and sharing 
  2025  Public perception on climate change: Analyzing impact of IPCC reports vi
  2025  AI-Driven Framework for Evaluating Climate Misinformation and Data Quali
  2026  SABER: Integrating sentiment and stance detection for climate change dis
  2022  Truth in an Age of Information
  2026  Fighting disinformation with AI tools: datasets for detecting fake news 
  2024  Climate Change Skeptics and the Power of Negativity
  2025  A Role for Artificial Intelli

### Preprints worth swapping

Items whose attached PDF looks like a preprint rather than the version of record. These
are flagged for the manual verification pass — if the published version is available, save
it over the preprint and re-run the reconciliation cell.

In [24]:
preprints = tracking[tracking["is_preprint"]]
print(f"{len(preprints)} PDFs look like preprints:\n")
for _, row in preprints.iterrows():
    print(f"  {row['title'][:72]}")
    print(f"      doi: {row['doi'] or '-'}   arxiv: {row['arxiv_id'] or '-'}\n")

2 PDFs look like preprints:

  LLM-Supported Content Analysis of Motivated Reasoning on Climate Change
      doi: 10.1002/pra2.1261   arxiv: 2508.21305

  Misinformation: determinants of gullibility
      doi: 10.31235/osf.io/r3fx7   arxiv: nan



### Match quality

Fuzzy title matches are the ones that could in principle be wrong, so they are worth a
glance. A mismatch here means a paper is reported as retrieved when it is not.

In [28]:
fuzzy = tracking[tracking["matched_by"].astype(str).str.startswith("title")]
if fuzzy.empty:
    print("Every match was on DOI or arXiv ID; nothing to check by hand.")
else:
    print(f"{len(fuzzy)} papers matched on title alone:\n")
    for _, row in fuzzy.iterrows():
        print(f"  matched_by : {row['matched_by']}")
        print(f"  screening  : {row['title']}")
        print(f"  zotero     : {str(row['zotero_title'])}\n")

6 papers matched on title alone:

  matched_by : title (93)
  screening  : A Multi-task Model for Sentiment Aided Stance Detection of Climate Change Tweets
  zotero     : A Multi-task Model for Emotion and Offensive Aided Stance Detection of Climate Change Tweets

  matched_by : title (100)
  screening  : Enhancing LLMs for Governance with Human Oversight: Evaluating and Aligning LLMs on Expert Classification of Climate Misinformation for Detecting False or Misleading Claims about Climate Change
  zotero     : Enhancing LLMs for Governance with Human Oversight: Evaluating and Aligning LLMs on Expert Classification of Climate Misinformation for Detecting False or Misleading Claims about Climate Change

  matched_by : title (100)
  screening  : WorthIt: Check-worthiness Estimation of Italian Social Media Posts
  zotero     : WorthIt: Check-worthiness Estimation of Italian Social Media Posts

  matched_by : title (100)
  screening  : TeamX at CheckThat! 2023: Multilingual and Multimodal A

## Steps 5 : Manual retrieval and verification

We use the `manual_worklist.html` on the VPN to go through the remaining papers, saving each paper into the same Zotero collection with the browser connector and checking all possible problems flagged in the section above. 

Whatever is still missing when you stop is not a failure — it is the PRISMA *reports not
retrieved* count, and `retrieval_tracking.xlsx` is the evidence for it. 

## Step 7 – reconcile again

We check again after doing the manual inspections and retrievals. 
In this case, also check if there are any papers that appear in the Zotero collection but are not in the screening list.

In [52]:
from src.zotero_retrieval import (
    AUDIT_PATH,
    audit_collection,
    summarise_audit,
    value,
    write_audit,
    write_worklist,
)

COLLECTION = "SLR_included"   # must match the collection name in Zotero exactly
tracking = reconcile(COLLECTION)

audit = audit_collection(COLLECTION)

print(summarise(audit.tracking))
print()
print(summarise_audit(audit))

# Refresh the forward-direction outputs too, so every file reflects this same read.
audit.tracking.to_excel(TRACKING_PATH, index=False)
write_worklist(audit.tracking)
write_audit(audit)

print(f"\nTracking sheet : {TRACKING_PATH}")
print(f"Worklist       : {WORKLIST_PATH}")
print(f"Audit sheet    : {AUDIT_PATH}")

Included papers               : 81
  present in Zotero           :  81 (100%)
  with a PDF attached         :  81 (100%)
  PDF looks like a preprint   :   0
  missing from Zotero         :   0
  in Zotero but no PDF        :   0

Items in the Zotero collection    : 81
  matching an included paper      :  81
  matching an EXCLUDED paper      :   0  <- remove from Zotero
  matching no screened paper      :   0
    of which look like duplicates :   0

Included papers not in Zotero     : 0

Tracking sheet : data/processed/retrieval/retrieval_tracking.xlsx
Worklist       : data/processed/retrieval/manual_worklist.html
Audit sheet    : data/processed/retrieval/collection_audit.xlsx


In [53]:
def titles(frame, column):
    """Titles from an audit frame, tolerating an empty frame that has no columns."""
    if frame.empty or column not in frame.columns:
        return []
    return list(frame[column])

not_included = titles(audit.excluded_present, "zotero_title") + titles(audit.unknown, "title")

print(f"{len(not_included)} items in the collection are not included papers:\n")
for title in not_included:
    print(f"  {title}")

0 items in the collection are not included papers:



## Conclusion: We found the PDFs for all the papers that passed the screening stage :)